## <b><font color='darkblue'>Preface</font></b>
([source](https://cloud.google.com/blog/topics/developers-practitioners/where-to-use-sub-agents-versus-agents-as-tools/)) <font size='3ptx'><b>At its simplest, an agent is an application that reasons on how to best achieve a goal based on inputs and tools at its disposal.</b></font>

![poc](https://storage.googleapis.com/gweb-cloudblog-publish/images/image1_oGjJbVH.max-800x800.png)

<b>As you build sophisticated multi-agent AI systems with the [Agent Development Kit](https://adk.dev/) (ADK), a key architectural decision involves choosing between a sub-agent and an agent as a tool. This choice fundamentally impacts your system's design, how well it scales, and its efficiency.</b> Choosing the wrong pattern can lead to massive overhead — either by constantly passing full conversational history to a simple function or by under-utilizing the context-sharing capabilities of a more complex system.

While both sub-agents and tools help break down complex problems, they serve different purposes. <b>The key difference is how they handle control and context.</b>

### <b><font color='darkgreen'>Agents as tools: The specialist on call</font></b>
<font size='3ptx'><b>An agent as a tool is a self-contained expert agent packaged for a specific, discrete task, like a specialized function call.</b> The main agent calls the tool with a clear input and gets a direct output, operating like a transactional API. The main agent doesn't need to worry about how the tool works; it only needs a reliable result. This pattern is ideal for independent and reusable tasks.</font>

Key characteristics:
- **Encapsulated and reusable:** The internal logic is hidden, making the tool easy to reuse across different agents.
- **Isolated context:** The tool runs in its own session and cannot access the calling agent’s conversation history or state.
- **Stateless:** The interaction is stateless. The tool receives all the information it needs in a single request.
- **Strict input/output**: It operates based on a well-defined contract.

### <b><font color='darkgreen'>Sub-agents: The delegated team member</font></b>
<font size='3ptx'><b>A sub-agent is a delegated team member that handles a complex, multi-step process.</b> This is a hierarchical and collaborative relationship where the sub-agent works within the broader context of the parent agent's mission.</font>

<b>Use sub-agents for tasks that require a chain of reasoning or a series of interactions.</b>

Key characteristics:
- **Tightly coupled and integrated**: Sub-agents are part of a larger, defined workflow.
- **Shared context:** They operate within the same session and can access the parent's conversation history and state, allowing for more nuanced collaboration.
- **Stateful processes**: They are ideal for managing processes where the task requires several steps to complete.
- **Hierarchical delegation**: The parent agent explicitly delegates a high-level task and lets the sub-agent manage the process.

### <b><font color='darkgreen'>Decision Making Table</font></b>
<font size='3ptx'>Here is a simple decision matrix that you can use to guide your architectural decision based on the task:</font>

| Criterion | Agent as a tool | Sub-agent | Decision |
| :--- | :--- | :--- | :--- |
| **Task complexity** | Low to Medium | High | Use a **tool** for atomic functions. Use a **sub-agent** for complex workflows. |
| **Context & state** | Isolated/None | Shared | If the task is stateless, use a **tool**. If it requires conversational context, use a **sub-agent**. |
| **Reusability** | High | Low to Medium | For generic, widely applicable capabilities, build a **tool**. For specialized roles in a specific process, use a **sub-agent**. |
| **Autonomy & control** | Low | High | Use a **tool** for a simple request-response. Use a **sub-agent** for delegating a whole sub-problem. |

## <b><font color='darkblue'>Use cases in action</font></b>
<font size='3ptx'>Let's apply this framework to some real-world scenarios. </font>

### <b><font color='darkgreen'>Use case 1: The data agent (NL2SQL and visualization)</font></b>
<font size='3ptx'>A business user asks for the top 5 product sales in Q2 by region and wants a bar chart. </font>

- <font size='3ptx'>**Root Agent:**</font> Receives the business user's request (NL), determines the necessary steps (<font color='brown'>SQL generation → Execution → Visualization</font>), and delegates/sequences the tasks, before returning the response to the user.
- <font size='3ptx'>**NL2SQL Agent**</font>: Use a tool. The task is a single, reusable function: convert natural language to a SQL string, using metadata & schema for grounding.
- <font size='3ptx'>**Database Executor**</font>: Use a tool. This is a simple, deterministic function to execute the query and return data.
- <font size='3ptx'>**Data Visualization Agent**</font>: <font color='blue'>Use a sub-agent. The task is complex and multi-step.</font> It involves analyzing the data returned by the database tool, and the original user query, selecting the right chart type, generating the visualization code, and executing it. Delegating this to a sub-agent allows the main orchestrator agent to maintain a high-level view while the sub-agent independently manages its complex internal workflow.

### <b><font color='darkgreen'>Use case 2: The sophisticated travel planner</font></b>
<font size='3ptx'>A user asks to plan a 5-day anniversary trip to Paris, with specific preferences for flights, hotels, and activities. This is an ambiguous, high-level goal that requires continuous context and planning. </font>

You could implement a Context/Memory Manager Tool accessible to all agents, potentially using a simple key-value store (like Redis or a simple database) to delegate the storage of immutable decisions. 

* <font size='3ptx'>**Travel planner**</font>: Use a root agent, to maintain the overall goal (<font color='brown'>"5-day anniversary trip to Paris"</font>), manage the flow between sub-agents, and aggregate the final itinerary.
* <font size='3ptx'>**Flight search**</font>: <font color='blue'>Use a sub-agent. The task is not a simple search; involving multiple back-and-forth interactions with the user</font> (<font color='brown'>e.g., "Is a layover in Dubai okay?"</font>) while managing the overall trip context (<font color='brown'>dates, destination, class</font>).
* <font size='3ptx'>**Hotel booking**</font>: <font color='blue'>Use a sub-agent. It needs to maintain state and context</font> (<font color='brown'>dates, location preference, 5-star rating</font>) as it searches for and presents options.
* <font size='3ptx'>**Itinerary generation**</font>: <font color='blue'>Use a sub-agent to generate a logical, day-by-day itinerary. The agent must combine confirmed flights/hotels with user interests</font> (<font color='brown'>e.g., art museums, fine dining</font>), potentially using its own booking tools.

<b>Using tools is inefficient; each call requires the full trip context, leading to redundancy and state loss</b>. Sub-agents are better for these stateful, collaborative processes as they share session context.

## <b><font color='darkblue'>Get started</font></b>
<font size='3ptx'>Let's use a simple example to learn the usage of sub-agent and agent as tool.</font>

In [32]:
from typing import Any
import json
from google.adk.agents import LlmAgent, BaseAgent
from google.adk.tools import agent_tool

from IPython.display import Image, display


FLASH_MODEL = "gemini-3.5-flash"

### <b><font color='darkgreen'>Testing Data - Employee CSV</font></b>
<font size='3ptx'>To test our SQL agent, we repare testing data saved in `employees.csv`:</font>

In [1]:
import sqlite3
import pandas as pd


TEST_CSV_DATA_FILE = 'employees.csv'

In [4]:
# 1. Load CSV into a Pandas DataFrame
df = pd.read_csv(TEST_CSV_DATA_FILE)

# 2. Connect to a temporary, in-memory SQLite database
conn = sqlite3.connect(':memory:')

# 3. Write the DataFrame to the SQL database
df.to_sql('Employee', conn, index=False, if_exists='replace')

200

In [3]:
df.head()

,Name,Age,Sex,Dept.,Salary,Seniority
0,James Rodriguez,37,M,Sales,69679,8
1,Andrew Wilson,24,M,Engineering,67082,1
2,Steven Martin,23,M,Sales,66428,5
3,Jessica Anderson,59,F,Engineering,94115,9
4,Lisa Rodriguez,31,F,Marketing,55674,4


In [7]:
test_sql = 'SELECT * FROM Employee WHERE Salary > 70000'
result_df = pd.read_sql_query(test_sql, conn)
result_df

,Name,Age,Sex,Dept.,Salary,Seniority
0,Jessica Anderson,59,F,Engineering,94115,9
1,Matthew Brown,44,M,Finance,96333,12
2,Mark Moore,29,M,Engineering,93544,7
3,Joseph Martinez,27,M,Engineering,80227,4
4,Jennifer Martin,62,F,Finance,75010,6
...,...,...,...,...,...,...
118,Linda Brown,63,F,Finance,127022,21
119,Margaret Davis,48,F,HR,85032,11
120,Linda Hernandez,49,F,Operations,82676,11
121,Linda Lopez,30,F,Engineering,101108,9


In [11]:
cursor = conn.cursor()
cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name='Employee';")
EMPLOYEE_TABLE_SCHEMA = cursor.fetchone()[0]
print(EMPLOYEE_TABLE_SCHEMA)

CREATE TABLE "Employee" (
"Name" TEXT,
  "Age" INTEGER,
  "Sex" TEXT,
  "Dept." TEXT,
  "Salary" INTEGER,
  "Seniority" INTEGER
)


### <b><font color='darkgreen'>Text-to-SQL Translator</font></b>

In [18]:
# Sub-Agent 1: Text-to-SQL Translator
sql_generator_agent = LlmAgent(
    name="SQL_Generator",
    model=FLASH_MODEL,
    description="Translate natural language into SQL.",
    instruction=f"""
    You are an expert SQL assistant. Your job is to translate natural language questions into valid SQLite queries.
    The database contains a table named 'Employee' with the following schema:
    {EMPLOYEE_TABLE_SCHEMA}
    
    Respond ONLY with the executable SQL query string. Do not include markdown formatting or explanations.
    """
)

### <b><font color='darkgreen'>Function to execute SQL query</font></b>

In [25]:
def execute_sql_query(sql_query: str, conn: Any = conn) -> str:
    """
    Executes a given SQL query string on the Employee database and returns the result as a JSON string.
    
    Args:
        sql_query (str): The SQL query string to be executed.
        conn (Any): An active database connection object (e.g., sqlite3 or SQLAlchemy connection).
        
    Returns:
        str: A JSON-formatted string containing the query results or error details.
    """
    try:
        # Execute query and load into a pandas DataFrame
        df_result = pd.read_sql_query(sql_query, conn)
        
        # Returns a JSON string representation of list of dictionaries
        return df_result.to_json(orient="records")
        
    except Exception as e:
        # Return a structured JSON error string so the LLM or ADK environment 
        # can predictably parse it without breaking schema expectations.
        return json.dumps({
            "status": "error",
            "message": f"Error executing query: {str(e)}"
        })

In [26]:
execute_sql_query('SELECT * FROM Employee WHERE Age > 40 LIMIT 20')

'[{"Name":"Jessica Anderson","Age":59,"Sex":"F","Dept.":"Engineering","Salary":94115,"Seniority":9},{"Name":"Matthew Brown","Age":44,"Sex":"M","Dept.":"Finance","Salary":96333,"Seniority":12},{"Name":"Michelle Lopez","Age":58,"Sex":"F","Dept.":"Operations","Salary":65639,"Seniority":7},{"Name":"Jennifer Martin","Age":62,"Sex":"F","Dept.":"Finance","Salary":75010,"Seniority":6},{"Name":"Jennifer Miller","Age":58,"Sex":"F","Dept.":"Marketing","Salary":123983,"Seniority":23},{"Name":"Donna Wilson","Age":59,"Sex":"F","Dept.":"Marketing","Salary":89093,"Seniority":13},{"Name":"Kimberly Rodriguez","Age":57,"Sex":"F","Dept.":"Operations","Salary":45376,"Seniority":1},{"Name":"Thomas Taylor","Age":60,"Sex":"M","Dept.":"Sales","Salary":97835,"Seniority":14},{"Name":"Charles Gonzalez","Age":64,"Sex":"M","Dept.":"Marketing","Salary":120677,"Seniority":21},{"Name":"Robert Johnson","Age":43,"Sex":"M","Dept.":"Finance","Salary":64399,"Seniority":3},{"Name":"Ashley Miller","Age":56,"Sex":"F","Dept.":

### <b><font color='darkgreen'>Chart Generator</font></b>

In [31]:
# 1. Refine the tool definition
def display_image(image_file_path: str) -> str:
    """
    Renders or registers a generated chart/image file so it can be viewed by the user.

    Args:
        image_file_path (str): The absolute file path to the saved image (e.g., '/tmp/chart.png').

    Returns:
        str: A confirmation status message verifying the file path is valid.
    """
    if not os.path.exists(image_file_path):
        return f"Error: File not found at {image_file_path}"
        
    # In a real ADK, instead of notebook's display(), you might log it, 
    # upload it to an artifact store, or return a payload the UI understands.
    print(f"[ADK Tool] Displaying image located at: {image_file_path}")
    
    return json.dumps({
        "status": "success",
        "message": f"Image successfully loaded and rendered from {image_file_path}",
        "file_path": image_file_path
    })


# 2. Refine the Agent Definition with structured instructions
chart_generator_agent = LlmAgent(
    name="Chart_Generator",
    model=FLASH_MODEL,
    instruction="""
    You are a data visualization expert. Given a dataset (provided as a JSON string) and a user request, 
    your job is to generate Python code using matplotlib or seaborn to create a visually appealing chart.

    CRITICAL EXECUTION RULES:
    1. Label everything clearly: Ensure your code includes titles, X/Y axis labels, and legends where appropriate.
    2. Save the output: Your generated Python code must explicitly save the resulting figure as an image file into the '/tmp/' directory (e.g., `/tmp/generated_chart.png`).
    3. Do not leave plots open: Always include `plt.close()` or `plt.clf()` at the end of your script to prevent memory leaks in the backend environment.
    4. Call the Viewer Tool: Immediately after generating and saving the image file, you MUST invoke the `display_image` tool, passing the exact file path you saved the image to.
    """,
    tools=[display_image],
)

### <b><font color='darkgreen'>Define the Root Agent</font></b>
<font size='3ptx'>Let's define an agent which can generate SQL query by nature language and compose a chart to display the result:</font>

In [33]:
coordinator = LlmAgent(
    name="SQLCoordinator",
    model=FLASH_MODEL,
    instruction="TBD",
    description="Accept request, compose SQL to query the DB and generate the chart to display the result.",
    # allow_transfer=True is often implicit with sub_agents in AutoFlow
    sub_agents=[chart_generator_agent],
    tools=[
        execute_sql_query,
        agent_tool.AgentTool(agent=sql_generator_agent),
    ]
)

### <b><font color='darkgreen'>Initialize Memory, Session etc for running the agent.</font></b>